# AI Bootcamp - Day 8 Assignment
## End-to-End NLP Pipeline: From Raw Text to Feature Vectors

---

### Assignment Overview

You will build a **complete NLP pipeline** - exactly the kind illustrated at the end of today's lecture:

```
Raw Text -> Preprocessing -> Feature Extraction -> ML Algorithm -> Text Classification
```

**Dataset:** SMS Spam Collection - 5,574 real SMS messages labelled `spam` or `ham`.
URL: https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv

This is the *same spam problem* discussed in the Rule-Based, Statistical, and Neural NLP slides!

---

### Learning Objectives

- Identify key NLP challenges (ambiguity, irregularities, missing text phenomenon)
- Apply a full preprocessing pipeline: tokenization, stopword removal, stemming/lemmatization, POS tagging, NER
- Build and compare feature representations: BoW, TF-IDF, N-grams, Word Embeddings
- Understand the limitations of each representation (as discussed in class)
- Train and evaluate a text classification pipeline
- Visualise and interpret what your model learned

---

### Required Libraries
```
pandas, numpy, scikit-learn, nltk, spacy, gensim, matplotlib, seaborn, wordcloud
```

**Estimated Time:** 2-3 hours | **Difficulty:** Intermediate

> **How to use:** Every `# -- TODO --` block is where you write your code.
> Read the description, study the hints, check the references, then implement.
> Do NOT skip the written reflection cells - they reinforce the concepts!

In [ ]:
# Uncomment and run once if needed
# !pip install pandas numpy scikit-learn nltk spacy gensim matplotlib seaborn wordcloud
# !python -m spacy download en_core_web_sm

---
## Section 1: Data Acquisition and Exploration

### Background

As discussed in today's lecture, NLP systems process billions of unstructured text records every day.
The SMS Spam Collection is a perfect real-world example: short-form, full of abbreviations, slang,
and deliberate misspellings - exactly the challenges that make NLP hard.

The file is tab-separated (.tsv). Each row has two fields:
- **Column 1:** Label - `ham` (legitimate) or `spam`
- **Column 2:** The raw SMS message text

---
### Task 1.1 - Load the Dataset

Fetch the dataset from the URL below and load it into a Pandas DataFrame with columns `['label', 'text']`.

> **Hint:** Use `pd.read_csv(url, sep='\t', header=None, names=['label','text'])`
> pandas can read directly from a URL!

> **References:**
> - Pandas read_csv: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
> - UCI SMS Spam Collection: https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

# -- TODO ------------------------------------------------------------------
# 1. Load the dataset from DATA_URL into a DataFrame named df
#    Columns should be: ['label', 'text']
# 2. Print df.shape
# 3. Display the first 5 rows with df.head()
# --------------------------------------------------------------------------

df = None  # Replace this line


---
### Task 1.2 - Exploratory Data Analysis (EDA)

Before building any pipeline, always **understand your data**. Answer the following:

1. How many `spam` vs `ham` messages are there? Is the dataset **balanced**?
2. What is the **average character length** of spam vs ham messages?
3. Plot a **bar chart** of label distribution
4. Plot a **histogram** of message lengths, coloured by label

> **Hint:** Create `df['text_length'] = df['text'].apply(len)` first.
> Use `seaborn.histplot(data=df, x='text_length', hue='label', bins=50)`

> **References:**
> - Seaborn histplot: https://seaborn.pydata.org/generated/seaborn.histplot.html
> - Pandas value_counts: https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html

> **Think about it:** If the dataset is imbalanced, should you use accuracy or F1 score?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -- TODO ------------------------------------------------------------------
# 1. Print label counts (spam vs ham)
# 2. Print average message length per label
# 3. Bar chart of label distribution
# 4. Histogram of message lengths coloured by label
# --------------------------------------------------------------------------


**Reflection 1.2:** Describe what you notice about the length distribution of spam vs ham.
Does message length alone seem like a useful feature? Why or why not?

*Write your answer here.*

---
## Section 2: Text Preprocessing Pipeline

### Background - The NLP Preprocessing Pipeline

As covered in today's lecture, raw text is full of noise.
The standard NLP preprocessing pipeline:

| Step | What it does | Example |
|------|-------------|---------|
| Lowercasing | Normalise case | "FREE" -> "free" |
| Punctuation Removal | Strip non-alpha chars | "Hi!!!" -> "Hi" |
| Tokenization | Split into word tokens | "I love NLP" -> ["I","love","NLP"] |
| Stop Word Removal | Remove function words | ["I","love","NLP"] -> ["love","NLP"] |
| Stemming | Chop to stem (crude) | "changing" -> "chang" |
| Lemmatization | Dictionary base form | "changing" -> "change" |

> **References:**
> - NLTK Book Ch.3: https://www.nltk.org/book/ch03.html
> - NLTK Stemming & Lemmatization: https://www.nltk.org/howto/stem.html
> - Stanford NLP - Stemming vs Lemmatization: https://nlp.stanford.edu/IR-book/html/htmledition/stemming-and-lemmatization-1.html

In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

STOPWORDS  = set(stopwords.words('english'))
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("NLTK resources loaded. Sample stopwords:", list(STOPWORDS)[:10])

---
### Task 2.1 - Implement the Preprocessing Function

Implement `preprocess_text(text, use_stemming=False, use_lemmatization=True)` that:

1. Converts to **lowercase**
2. Removes **punctuation and numbers** (keep only letters and spaces)
3. **Tokenizes** using `word_tokenize()`
4. Removes **stop words**
5. Applies **stemming** OR **lemmatization** based on the flag
6. Returns cleaned tokens **joined as a single string**

> **Hint - Step by step:**
> ```python
> text = text.lower()
> text = re.sub(r'[^a-z\s]', '', text)
> tokens = word_tokenize(text)
> tokens = [w for w in tokens if w not in STOPWORDS and len(w) > 1]
> # apply stemmer.stem(w) or lemmatizer.lemmatize(w) to each token
> return ' '.join(tokens)
> ```

> **Recall from lecture:** Stemming uses crude chopping (`chang` for change/changing/changed),
> Lemmatization returns a proper dictionary word (`change`).
> Stemming is faster; Lemmatization is more accurate.

In [ ]:
def preprocess_text(text, use_stemming=False, use_lemmatization=True):
    """
    Full NLP preprocessing pipeline.

    Parameters:
        text (str): Raw input SMS text
        use_stemming (bool): Apply Porter stemming if True
        use_lemmatization (bool): Apply WordNet lemmatization if True

    Returns:
        str: Preprocessed text as a space-joined string of clean tokens
    """
    # -- TODO ------------------------------------------------------------------
    pass
    # --------------------------------------------------------------------------


# Test your function on these two examples
spam_sample = "FREE entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005! Call 08452810075!"
ham_sample  = "I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k?"

print("SPAM - Original :", spam_sample)
print("SPAM - Processed:", preprocess_text(spam_sample))
print()
print("HAM  - Original :", ham_sample)
print("HAM  - Processed:", preprocess_text(ham_sample))

---
### Task 2.2 - Apply Preprocessing to the Full Dataset

Apply `preprocess_text()` to the entire `text` column and store the result in `clean_text`.

> **Hint:** `df['clean_text'] = df['text'].apply(preprocess_text)`
> This may take 10-20 seconds for 5,574 messages.

Then print a side-by-side comparison for 2 spam + 2 ham messages.

In [ ]:
# -- TODO ------------------------------------------------------------------
# 1. Apply preprocess_text to df['text'] -> store in df['clean_text']
# 2. Show before/after for 2 spam + 2 ham messages
# --------------------------------------------------------------------------


---
### Task 2.3 - WordCloud: Spam vs Ham

Generate two WordClouds side by side using `clean_text`.

This gives visual intuition for which words associate with each class -
directly connecting to the Rule-Based NLP slide (R2: words like "lottery" and "dollars"...)

> **Hint:**
> ```python
> spam_text = ' '.join(df[df['label'] == 'spam']['clean_text'])
> wc = WordCloud(width=600, height=400, background_color='white').generate(spam_text)
> plt.imshow(wc, interpolation='bilinear')
> ```
> Reference: https://amueller.github.io/word_cloud/

In [ ]:
from wordcloud import WordCloud

# -- TODO ------------------------------------------------------------------
# 1. Join all spam clean_text into one string
# 2. Join all ham clean_text into one string
# 3. Generate and display WordClouds side by side (plt.subplot(1, 2, ...))
# --------------------------------------------------------------------------


---
### Task 2.4 - POS Tagging and Named Entity Recognition (NER)

The lecture covered POS tagging and NER as key NLP tools. Let's see them with spaCy.

1. Load the `en_core_web_sm` spaCy model
2. Pick one spam and one ham message (use raw `text`, not `clean_text`)
3. For each message, print all **tokens with their POS tags**
4. For each message, print all **named entities** found

> **Hint:**
> ```python
> import spacy
> nlp = spacy.load('en_core_web_sm')
> doc = nlp("Your text here")
> for token in doc:
>     print(token.text, token.pos_)    # POS tagging
> for ent in doc.ents:
>     print(ent.text, ent.label_)      # NER
> ```

> **References:**
> - spaCy POS tagging: https://spacy.io/usage/linguistic-features#pos-tagging
> - spaCy NER: https://spacy.io/usage/linguistic-features#named-entities
> - UD POS tag meanings: https://universaldependencies.org/u/pos/

In [ ]:
import spacy

# Load model (run: python -m spacy download en_core_web_sm  if not installed)
nlp = spacy.load('en_core_web_sm')

spam_msg = df[df['label'] == 'spam']['text'].iloc[0]
ham_msg  = df[df['label'] == 'ham']['text'].iloc[5]

# -- TODO ------------------------------------------------------------------
# 1. For spam_msg: print all tokens with POS tags, then named entities
# 2. For ham_msg:  print all tokens with POS tags, then named entities
# --------------------------------------------------------------------------


**Reflection 2.4:** Did the NER model find entities in the spam message?
What types (PERSON, ORG, MONEY, DATE...)? Could NER features help spam detection? Explain briefly.

*Write your answer here.*

---
## Section 3: Feature Extraction

### Background - Text as Vectors

As shown in today's lecture, ML algorithms cannot work on raw text -
text must be converted to numerical vectors. Four main strategies from class:

| Method | Idea | Strengths | Weaknesses |
|--------|------|-----------|------------|
| N-grams | Sequences of N consecutive words | Captures local word order | Vocabulary explosion |
| Bag of Words (BoW) | Count each word's frequency | Simple, interpretable | Ignores order |
| TF-IDF | Weight words by uniqueness across docs | Highlights distinctive words | Ignores semantics |
| Word Embeddings | Dense semantic vectors (Word2Vec, GloVe, FastText) | Captures meaning | Needs large data |

> **Key References:**
> - Scikit-learn text feature extraction: https://scikit-learn.org/stable/modules/feature_extraction.html
> - TF-IDF explained: https://monkeylearn.com/blog/what-is-tf-idf/
> - Word2Vec paper (Mikolov 2013): https://arxiv.org/abs/1301.3781
> - GloVe pretrained vectors: https://nlp.stanford.edu/projects/glove/

---
### Task 3.1 - Bag of Words (BoW)

Use `CountVectorizer` to build a Bag of Words representation.

1. Fit and transform `df['clean_text']` into a BoW matrix
2. Print the total **vocabulary size**
3. Print the **top 20 most frequent words** across the entire corpus
4. Re-run with `max_features=500` - what changes?

> **Hint:**
> ```python
> from sklearn.feature_extraction.text import CountVectorizer
> cv = CountVectorizer()
> bow_matrix = cv.fit_transform(df['clean_text'])
> vocab      = cv.get_feature_names_out()
> word_freq  = bow_matrix.sum(axis=0).A1
> top_idx    = word_freq.argsort()[::-1][:20]
> ```

> **Recall the BoW limitation from lecture:** common words like 'call' score highly
> for BOTH spam and ham. Does word count alone discriminate well?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# -- TODO ------------------------------------------------------------------
# 1. CountVectorizer fit_transform on df['clean_text']
# 2. Print vocabulary size
# 3. Print top 20 most frequent words
# 4. Re-run with max_features=500 and compare
# --------------------------------------------------------------------------


---
### Task 3.2 - TF-IDF

Build TF-IDF vectors and explore how they improve on plain BoW.

Recall the formula from lecture:

    w(i,j) = tf(i,j) x log( N / df(i) )

A word common in every document gets a low IDF weight.

1. Fit `TfidfVectorizer(max_features=1000)` on `df['clean_text']`
2. For 3 randomly chosen spam messages, print the **top 10 TF-IDF scoring words**
3. Find a word that is **frequent in BoW but has low TF-IDF** - name it and explain why

> **Hint - Top words for one document:**
> ```python
> row    = tfidf_matrix[doc_index]
> scores = row.toarray()[0]
> top_idx = scores.argsort()[::-1][:10]
> top_words = [feature_names[i] for i in top_idx if scores[i] > 0]
> ```

> Reference: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# -- TODO ------------------------------------------------------------------
# 1. Fit TfidfVectorizer(max_features=1000) on df['clean_text']
# 2. For 3 spam messages, show the top-10 TF-IDF terms
# 3. Answer the comparison in the reflection cell below
# --------------------------------------------------------------------------


**Reflection 3.2:** Name one word frequent in BoW but with low TF-IDF weight.
Why does TF-IDF penalise it? How does this connect to the BoW Limitation slide from lecture?

*Write your answer here.*

---
### Task 3.3 - N-grams

As shown in lecture, N-grams capture word sequences.
Bigrams like "free entry" or "call now" are far more discriminative for spam than single words.

1. Create `TfidfVectorizer(ngram_range=(1,2), max_features=2000)` - includes unigrams and bigrams
2. Find the **top 20 bigrams** most associated with spam messages
3. Plot a horizontal bar chart of these spam bigrams

> **Hint - Isolating bigrams from the vocabulary:**
> ```python
> feature_names = tfidf_ngram.get_feature_names_out()
> bigram_mask = np.array([len(f.split()) == 2 for f in feature_names])
> ```
> Fit on all data, transform only spam subset, average column-wise.

> Reference: https://towardsdatascience.com/introduction-to-language-models-n-grams-10c3f6e3c0ad

In [ ]:
# -- TODO ------------------------------------------------------------------
# 1. Fit TfidfVectorizer(ngram_range=(1,2), max_features=2000)
# 2. Transform only spam messages and average TF-IDF per feature
# 3. Filter for bigrams only (feature name contains a space)
# 4. Plot top 20 spam bigrams as a horizontal bar chart
# --------------------------------------------------------------------------


---
### Task 3.4 - Word Embeddings with Word2Vec

From the lecture: "Word embedding is the representation of text in the form of vectors.
Similar words have minimum distance between their vectors."
Famous example: King - Man + Woman = Queen

**Part A - Train Word2Vec:**
1. Tokenize each `clean_text` into a list of words (list of lists)
2. Train `Word2Vec(vector_size=100, window=5, min_count=2)`
3. Find the top 10 words most similar to `'free'`
4. Find the top 10 words most similar to `'call'`

**Part B - Document vectors:**
Create a document embedding for each message by **averaging** its word vectors.

> **Hint - Train:**
> ```python
> from gensim.models import Word2Vec
> sentences = [text.split() for text in df['clean_text']]
> w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)
> w2v_model.wv.most_similar('free', topn=10)
> ```

> **Hint - Average pooling:**
> ```python
> def get_doc_vector(text, model, size=100):
>     words = text.split()
>     vecs  = [model.wv[w] for w in words if w in model.wv]
>     return np.mean(vecs, axis=0) if vecs else np.zeros(size)
> ```

> **References:**
> - Gensim Word2Vec: https://radimrehurek.com/gensim/models/word2vec.html
> - Illustrated Word2Vec: https://jalammar.github.io/illustrated-word2vec/
> - Pretrained GloVe: https://nlp.stanford.edu/projects/glove/

In [ ]:
from gensim.models import Word2Vec

# -- TODO - Part A: Train Word2Vec -----------------------------------------
# 1. Tokenize df['clean_text'] into sentences (list of word lists)
# 2. Train Word2Vec(vector_size=100, window=5, min_count=2)
# 3. Print top-10 words most similar to 'free'
# 4. Print top-10 words most similar to 'call'
# --------------------------------------------------------------------------


# -- TODO - Part B: Document vectors ---------------------------------------
# Implement get_doc_vector() using average pooling
# Create X_w2v: numpy array of shape (n_docs, 100)
# --------------------------------------------------------------------------

def get_doc_vector(text, model, size=100):
    # TODO: implement average pooling
    pass

# X_w2v = np.array([get_doc_vector(text, w2v_model) for text in df['clean_text']])
# print("Word2Vec feature matrix shape:", X_w2v.shape)

**Reflection 3.4:** Look at the words most similar to 'free' in the embedding space.
Do the neighbours make sense for the spam domain?
How is this different from what TF-IDF would tell you about the word 'free'?

*Write your answer here.*

---
## Section 4: Text Classification Pipeline

### Background

Today's lecture ended with the Text Classification Pipeline:

```
TEXT + TAG -> Feature Extraction -> FEATURES -> ML Algorithm -> Text Classification
```

You will now assemble this using `sklearn.pipeline.Pipeline`, which chains a vectorizer
and classifier into one reusable object. This also prevents **data leakage**:
the vectorizer must only be fit on training data, never the test set.

You will compare three approaches from today's lecture:
- **Statistical NLP:** TF-IDF + Naive Bayes (Bayes Theorem slide)
- **Better Statistical:** TF-IDF N-grams + Logistic Regression
- **Neural / Embedding:** Word2Vec features + Logistic Regression

> **References:**
> - sklearn Pipeline: https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
> - Naive Bayes for text: https://scikit-learn.org/stable/modules/naive_bayes.html
> - Why data leakage is dangerous: https://machinelearningmastery.com/data-leakage-machine-learning/
> - Classification metrics: https://scikit-learn.org/stable/modules/model_evaluation.html

---
### Task 4.1 - Encode Labels and Split Data

1. Encode labels: `ham -> 0`, `spam -> 1` using `LabelEncoder`
2. Split into **80% train / 20% test** with `stratify=y` and `random_state=42`
3. Print class distribution in both splits to confirm stratification

> **Hint:** `stratify=y` ensures the spam/ham ratio is preserved in both splits.
> Without it, you might accidentally put most spam in one split only!

> Reference: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# -- TODO ------------------------------------------------------------------
# 1. Encode labels: ham=0, spam=1 using LabelEncoder
# 2. Split: X=df['clean_text'], y=encoded labels
#    Use test_size=0.2, stratify=y, random_state=42
# 3. Print class distribution in train and test splits
# --------------------------------------------------------------------------

X = df['clean_text']
y = None  # encode labels here

# X_train, X_test, y_train, y_test = ...

---
### Task 4.2 - Build and Compare Three Pipelines

| Pipeline | Vectorizer | Classifier | Inspiration |
|----------|-----------|------------|-------------|
| A | TfidfVectorizer(max_features=1000) | MultinomialNB | Statistical NLP from lecture |
| B | TfidfVectorizer(max_features=2000, ngram_range=(1,2)) | LogisticRegression | N-gram TF-IDF |
| C | Word2Vec avg vectors from Task 3.4 | LogisticRegression | Neural NLP from lecture |

For each pipeline:
1. Fit on training data
2. Predict on test data
3. Print a **classification report** (precision, recall, F1)
4. Plot a **confusion matrix heatmap**

> **Hint - Pipeline A and B:**
> ```python
> from sklearn.pipeline import Pipeline
> pipe_a = Pipeline([
>     ('tfidf', TfidfVectorizer(max_features=1000)),
>     ('clf',   MultinomialNB())
> ])
> pipe_a.fit(X_train, y_train)
> y_pred = pipe_a.predict(X_test)
> ```

> **Hint - Pipeline C:** uses X_w2v (numpy array) directly, no vectorizer needed.
> Split X_w2v with the same indices as X_train/X_test.

> **Note on Naive Bayes:** This is the Bayes Theorem approach from the Statistical NLP slide:
> P(c|X) = P(x1|c) x P(x2|c) x ... x P(c)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# -- TODO - Pipeline A: TF-IDF + Naive Bayes --------------------------------


# -- TODO - Pipeline B: Bigram TF-IDF + Logistic Regression -----------------


# -- TODO - Pipeline C: Word2Vec + Logistic Regression ----------------------
# Split X_w2v using the same train/test indices


---
### Task 4.3 - Cross-Validation for Robust Evaluation

A single train/test split can be misleading. Use **5-fold cross-validation** for reliable estimates.

1. Run `cross_val_score` with `cv=5` and `scoring='f1'` for Pipeline A and B
2. Print **mean +/- standard deviation** for each
3. Which model is more **consistent** (lower std)? Which scores higher?

> **Hint:**
> ```python
> from sklearn.model_selection import cross_val_score
> scores = cross_val_score(pipe_a, X, y, cv=5, scoring='f1')
> print(f"Pipeline A: {scores.mean():.4f} +/- {scores.std():.4f}")
> ```

> Reference: https://scikit-learn.org/stable/modules/cross_validation.html

In [ ]:
from sklearn.model_selection import cross_val_score

# -- TODO ------------------------------------------------------------------
# 5-fold cross-validation for Pipeline A and Pipeline B
# Print mean +/- std for each
# --------------------------------------------------------------------------


---
## Section 5: Model Interpretation

### Background

Understanding WHY a model makes predictions is as important as the prediction itself.
For Logistic Regression, feature coefficients directly tell us which words push toward
spam or ham - bridging back to the Rule-Based NLP intuition from today's lecture.

> Reference: https://christophm.github.io/interpretable-ml-book/limo.html

---
### Task 5.1 - Top Predictive Features

From Pipeline B (Logistic Regression), extract and visualise the most predictive features.

1. Get feature names: `pipe_b['tfidf'].get_feature_names_out()`
2. Get coefficients: `pipe_b['clf'].coef_[0]`
3. Plot a **horizontal bar chart** showing:
   - Top 15 features most predictive of **spam** (highest positive coefficients)
   - Top 15 features most predictive of **ham** (most negative coefficients)
   - Use different colours for spam vs ham bars

> **Hint:**
> ```python
> feature_names = pipe_b['tfidf'].get_feature_names_out()
> coefs         = pipe_b['clf'].coef_[0]
> top_spam_idx  = coefs.argsort()[-15:]   # highest 15 = most spam-like
> top_ham_idx   = coefs.argsort()[:15]    # lowest 15  = most ham-like
> ```

In [ ]:
# -- TODO ------------------------------------------------------------------
# 1. Extract feature names and coefficients from pipe_b
# 2. Plot top-15 spam features and top-15 ham features as horizontal bar charts
# --------------------------------------------------------------------------


**Reflection 5.1:** Compare the top spam-predictive words your model found
vs the rules from the lecture slide (R1: misspellings, R2: words like "lottery"/"dollars",
R3: urgency, R4: inappropriate language).

Did your data-driven model independently rediscover these rules?
What does this tell you about the relationship between Rule-Based and Statistical NLP?

*Write your answer here.*

---
### Task 5.2 - Error Analysis

Find cases where the model made mistakes.

1. Find **5 False Positives** (ham predicted as spam) from Pipeline B
2. Find **5 False Negatives** (spam predicted as ham)
3. Display the **original raw text** for each
4. Try to explain WHY the model was confused

> **Hint:**
> ```python
> test_df = df.loc[X_test.index].copy()
> test_df['predicted'] = y_pred_b
> test_df['actual']    = y_test.values
> false_pos = test_df[(test_df['actual']==0) & (test_df['predicted']==1)]
> false_neg = test_df[(test_df['actual']==1) & (test_df['predicted']==0)]
> ```

In [ ]:
# -- TODO ------------------------------------------------------------------
# 1. Find 5 false positives (ham predicted as spam)
# 2. Find 5 false negatives (spam predicted as ham)
# 3. Display original raw text for each case
# --------------------------------------------------------------------------


**Reflection 5.2:** For one false positive and one false negative,
explain why the model made a mistake. Refer to specific NLP challenges from the lecture
(ambiguity, idioms, missing text phenomenon, sarcasm, etc.)

*Write your answer here.*

---
## Section 6: Bonus Challenges

Completed the core tasks? Try these extensions!

---
### Bonus 1 - Semantic Similarity with Word2Vec

The lecture showed: King - Man + Woman = Queen. Try it on your trained model.

Use `w2v_model.wv.most_similar(positive=['winner', 'prize'], negative=['free'], topn=5)`

Does vector arithmetic work as well on an SMS dataset vs Wikipedia?
Why or why not? Think about vocabulary size and data volume.

In [ ]:
# -- BONUS 1 - Word vector arithmetic -------------------------------------
# Try: winner - prize + free = ?
# Try your own analogies relevant to the spam/ham domain
# --------------------------------------------------------------------------


---
### Bonus 2 - Hyperparameter Tuning with GridSearchCV

Use `GridSearchCV` to find the best parameters for Pipeline B:

```python
param_grid = {
    'tfidf__max_features': [500, 1000, 2000],
    'tfidf__ngram_range':  [(1,1), (1,2)],
    'clf__C':              [0.1, 1.0, 10.0]
}
```

Reference: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [ ]:
from sklearn.model_selection import GridSearchCV

# -- BONUS 2 - GridSearchCV on Pipeline B ----------------------------------
# Use cv=3 to save time. Print best_params_ and best_score_
# --------------------------------------------------------------------------


---
### Bonus 3 - Stemming vs Lemmatization: Does It Matter?

Re-run Pipeline B twice:
- Once with `use_stemming=True, use_lemmatization=False`
- Once with default (lemmatization)

Compare F1 scores. Which is better for this dataset?
Is the difference significant? Why might that be?

In [ ]:
# -- BONUS 3 - Stemming vs Lemmatization comparison -----------------------


### Additional Learning Resources

| Topic | Resource |
|-------|----------|
| Full NLTK reference | https://www.nltk.org/book/ |
| spaCy course | https://course.spacy.io |
| Lena Voita's NLP Course (cited in lecture) | https://lena-voita.github.io/nlp_course.html |
| Stanford CS224N NLP with Deep Learning | https://web.stanford.edu/class/cs224n/ |
| Illustrated Word2Vec | https://jalammar.github.io/illustrated-word2vec/ |
| Gensim Word2Vec | https://radimrehurek.com/gensim/models/word2vec.html |
| Scikit-learn Text Feature Extraction | https://scikit-learn.org/stable/modules/feature_extraction.html |

---
*Assignment prepared for AI Bootcamp 2026 - Day 8: Introduction to Natural Language Processing*
*Instructor: Dr. Shailesh S, CUSAT*